# Construccion de canasta de precios
A partir de datos obtenidos de [SEPA](https://datos.produccion.gob.ar/dataset/sepa-precios) se busca obtener una lista de productos unicos y evaluar su representatividad

In [1]:
# ============================================================
# CELDA 1 — Configuración e imports
# ============================================================
import os
import zipfile
import glob
import io
import gc
import math
import pandas as pd
import numpy as np
from collections import defaultdict
from pathlib import Path

# >>> AJUSTAR SOLO ESTA LÍNEA <
ZIP_DIA = "/content/2026-05-12.zip"

WORK_DIR = "/content/sepa_work"
OUTPUT_XLSX = "/content/productos_unicos_resumen.xlsx"

os.makedirs(WORK_DIR, exist_ok=True)
print("OK — configuración lista")

OK — configuración lista


In [2]:
# ============================================================
# CELDA 2 — Descomprimir el zip del día y listar los zips internos
# ============================================================
with zipfile.ZipFile(ZIP_DIA, 'r') as z:
    z.extractall(WORK_DIR)

zips_cadenas = sorted(glob.glob(os.path.join(WORK_DIR, "**", "sepa_*.zip"), recursive=True))

carpeta_dia = None
for root, dirs, files in os.walk(WORK_DIR):
    for d in dirs:
        if len(d) == 10 and d[4] == '-' and d[7] == '-':
            carpeta_dia = d
            break
    if carpeta_dia:
        break

print(f"Fecha detectada: {carpeta_dia}")
print(f"Cantidad de ZIPs de cadenas encontrados: {len(zips_cadenas)}")
for z in zips_cadenas[:5]:
    print(" -", os.path.basename(z))

Fecha detectada: None
Cantidad de ZIPs de cadenas encontrados: 20
 - sepa_1_comercio-sepa-11_2026-05-12_09-05-10.zip
 - sepa_1_comercio-sepa-12_2026-05-12_09-05-10.zip
 - sepa_1_comercio-sepa-13_2026-05-12_09-05-10.zip
 - sepa_1_comercio-sepa-15_2026-05-12_09-05-10.zip
 - sepa_1_comercio-sepa-16_2026-05-12_09-05-10.zip


In [3]:
# ============================================================
# CELDA 3 — Función para leer CSVs SEPA desde un zip
# ============================================================
def leer_csv_sepa(zip_path, nombre_csv, columnas=None):
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            target = None
            for name in z.namelist():
                if name.lower().endswith(nombre_csv.lower()):
                    target = name
                    break
            if target is None:
                return None

            with z.open(target) as f:
                contenido = f.read()

        for enc in ['utf-8', 'latin-1']:
            try:
                texto = contenido.decode(enc)
                break
            except UnicodeDecodeError:
                continue
        del contenido

        lineas = [l for l in texto.split('\n')
                  if l.strip()
                  and not l.lower().startswith('última actualización')
                  and not l.lower().startswith('ultima actualizacion')
                  and not l.lower().startswith('\ufeffúltima')]
        texto_limpio = '\n'.join(lineas)
        del texto, lineas

        df = pd.read_csv(io.StringIO(texto_limpio),
                         sep='|',
                         dtype=str,
                         usecols=columnas,
                         on_bad_lines='skip')
        return df
    except Exception as e:
        print(f"  ⚠️ Error leyendo {nombre_csv} en {os.path.basename(zip_path)}: {e}")
        return None

print("OK — función definida")

OK — función definida


In [4]:
# ============================================================
# CELDA 4 — Procesar zip por zip y AGREGAR incrementalmente
# ============================================================
MAESTRO_PROV = {
    'AR-C':'CABA','AR-B':'Buenos Aires','AR-K':'Catamarca','AR-H':'Chaco',
    'AR-U':'Chubut','AR-X':'Córdoba','AR-W':'Corrientes','AR-E':'Entre Ríos',
    'AR-P':'Formosa','AR-Y':'Jujuy','AR-L':'La Pampa','AR-F':'La Rioja',
    'AR-M':'Mendoza','AR-N':'Misiones','AR-Q':'Neuquén','AR-R':'Río Negro',
    'AR-A':'Salta','AR-J':'San Juan','AR-D':'San Luis','AR-Z':'Santa Cruz',
    'AR-S':'Santa Fe','AR-G':'Santiago del Estero','AR-V':'Tierra del Fuego',
    'AR-T':'Tucumán',
}

COLS_PROD = ['id_comercio','id_bandera','id_sucursal','id_producto',
             'productos_ean','productos_descripcion','productos_marca',
             'productos_cantidad_presentacion','productos_unidad_medida_presentacion',
             'productos_precio_lista']
COLS_SUC = ['id_comercio','id_bandera','id_sucursal',
            'sucursales_provincia','sucursales_tipo','sucursales_localidad']
COLS_COM = ['id_comercio','id_bandera','comercio_bandera_nombre']

acc = defaultdict(lambda: {
    'descripcion': None, 'marca': None,
    'cantidad': None, 'unidad': None,
    'es_ean_oficial': False,
    'n_registros': 0,
    'sucursales': set(),
    'cadenas': set(),
    'provincias': set(),
    'suma_precio': 0.0,
    'suma_precio2': 0.0,
    'n_precio': 0,
    'min_precio': float('inf'),
    'max_precio': float('-inf'),
})

cob_prov = defaultdict(lambda: {'productos':set(),'sucursales':set(),'cadenas':set(),'n_reg':0})
cob_cad  = defaultdict(lambda: {'productos':set(),'sucursales':set(),'provincias':set(),'n_reg':0})
matriz_cad_prov = defaultdict(int)

total_filas_productos = 0

for i, zp in enumerate(zips_cadenas, 1):
    nombre = os.path.basename(zp)
    print(f"[{i}/{len(zips_cadenas)}] {nombre}")

    df_com = leer_csv_sepa(zp, 'comercio.csv', columnas=COLS_COM)
    df_suc = leer_csv_sepa(zp, 'sucursales.csv', columnas=COLS_SUC)
    df_pro = leer_csv_sepa(zp, 'productos.csv', columnas=COLS_PROD)

    if df_pro is None or df_suc is None or df_com is None:
        print("  ⚠️ Faltan archivos, salteando")
        continue

    df_com = df_com.drop_duplicates(subset=['id_comercio','id_bandera'])
    cadena_lookup = {(r.id_comercio, r.id_bandera): r.comercio_bandera_nombre
                     for r in df_com.itertuples(index=False)}

    suc_lookup = {}
    for r in df_suc.itertuples(index=False):
        key = (r.id_comercio, r.id_bandera, r.id_sucursal)
        suc_lookup[key] = r.sucursales_provincia

    df_pro['precio'] = pd.to_numeric(df_pro['productos_precio_lista'], errors='coerce')
    total_filas_productos += len(df_pro)

    for r in df_pro.itertuples(index=False):
        ean = r.id_producto
        if ean is None or (isinstance(ean, float) and np.isnan(ean)):
            continue

        suc_key = (r.id_comercio, r.id_bandera, r.id_sucursal)
        prov_iso = suc_lookup.get(suc_key)
        provincia = MAESTRO_PROV.get(prov_iso) if prov_iso else None
        cadena = cadena_lookup.get((r.id_comercio, r.id_bandera))

        a = acc[ean]
        if a['descripcion'] is None and r.productos_descripcion:
            a['descripcion'] = r.productos_descripcion
            a['marca'] = r.productos_marca
            a['cantidad'] = r.productos_cantidad_presentacion
            a['unidad'] = r.productos_unidad_medida_presentacion

        if r.productos_ean == '1':
            a['es_ean_oficial'] = True

        a['n_registros'] += 1
        a['sucursales'].add(suc_key)
        if cadena: a['cadenas'].add(cadena)
        if provincia: a['provincias'].add(provincia)

        precio = r.precio
        if precio is not None and not np.isnan(precio) and precio > 0:
            a['suma_precio'] += precio
            a['suma_precio2'] += precio * precio
            a['n_precio'] += 1
            if precio < a['min_precio']: a['min_precio'] = precio
            if precio > a['max_precio']: a['max_precio'] = precio

        if provincia:
            cp = cob_prov[provincia]
            cp['productos'].add(ean)
            cp['sucursales'].add(suc_key)
            if cadena: cp['cadenas'].add(cadena)
            cp['n_reg'] += 1

        if cadena:
            cc = cob_cad[cadena]
            cc['productos'].add(ean)
            cc['sucursales'].add(suc_key)
            if provincia: cc['provincias'].add(provincia)
            cc['n_reg'] += 1

            if provincia:
                matriz_cad_prov[(cadena, provincia)] += 1

    del df_com, df_suc, df_pro
    gc.collect()

print(f"\n✅ Procesamiento completo")
print(f"Total filas de productos procesadas: {total_filas_productos:,}")
print(f"Productos únicos (EANs distintos): {len(acc):,}")

[1/20] sepa_1_comercio-sepa-11_2026-05-12_09-05-10.zip
[2/20] sepa_1_comercio-sepa-12_2026-05-12_09-05-10.zip
[3/20] sepa_1_comercio-sepa-13_2026-05-12_09-05-10.zip
[4/20] sepa_1_comercio-sepa-15_2026-05-12_09-05-10.zip
[5/20] sepa_1_comercio-sepa-16_2026-05-12_09-05-10.zip
[6/20] sepa_1_comercio-sepa-19_2026-05-12_09-05-10.zip
[7/20] sepa_1_comercio-sepa-20_2026-05-12_09-05-10.zip
[8/20] sepa_1_comercio-sepa-21_2026-05-12_09-05-10.zip
[9/20] sepa_1_comercio-sepa-23_2026-05-12_09-05-10.zip
[10/20] sepa_1_comercio-sepa-24_2026-05-12_09-05-10.zip
[11/20] sepa_1_comercio-sepa-3_2026-05-12_09-05-10.zip
[12/20] sepa_1_comercio-sepa-47_2026-05-12_09-05-10.zip
[13/20] sepa_1_comercio-sepa-4_2026-05-12_09-05-10.zip
[14/20] sepa_1_comercio-sepa-8_2026-05-12_09-05-10.zip
[15/20] sepa_1_comercio-sepa-9_2026-05-12_09-05-10.zip
[16/20] sepa_2_comercio-sepa-10_2026-05-12_01-06-06.zip
[17/20] sepa_2_comercio-sepa-2_2026-05-12_01-06-06.zip
[18/20] sepa_2_comercio-sepa-36_2026-05-12_01-06-06.zip
[19/20

In [5]:
# ============================================================
# CELDA 5 — Construir DataFrame de resumen desde los acumuladores
# ============================================================
filas = []
for ean, a in acc.items():
    n = a['n_precio']
    if n > 0:
        media = a['suma_precio'] / n
        var = max(a['suma_precio2']/n - media*media, 0)
        std = math.sqrt(var)
        cv = std / media if media > 0 else None
        pmin = a['min_precio']
        pmax = a['max_precio']
    else:
        media = std = cv = pmin = pmax = None

    filas.append({
        'id_producto': ean,
        'descripcion': a['descripcion'],
        'marca': a['marca'],
        'cantidad_presentacion': a['cantidad'],
        'unidad_presentacion': a['unidad'],
        'es_ean_oficial': a['es_ean_oficial'],
        'n_registros': a['n_registros'],
        'n_sucursales': len(a['sucursales']),
        'n_cadenas': len(a['cadenas']),
        'n_provincias': len(a['provincias']),
        'cadenas_lista': ', '.join(sorted(a['cadenas'])),
        'provincias_lista': ', '.join(sorted(a['provincias'])),
        'precio_min': round(pmin, 2) if pmin is not None else None,
        'precio_max': round(pmax, 2) if pmax is not None else None,
        'precio_promedio': round(media, 2) if media is not None else None,
        'precio_cv': round(cv, 3) if cv is not None else None,
    })

resumen = pd.DataFrame(filas).sort_values(
    ['n_provincias','n_cadenas','n_registros'],
    ascending=[False, False, False]
).reset_index(drop=True)

del acc, filas
gc.collect()

print(f"Productos únicos: {len(resumen):,}")
resumen.head(15)

Productos únicos: 87,418


,id_producto,descripcion,marca,cantidad_presentacion,unidad_presentacion,es_ean_oficial,n_registros,n_sucursales,n_cadenas,n_provincias,cadenas_lista,provincias_lista,precio_min,precio_max,precio_promedio,precio_cv
0,7790895641183,POWERADE FRUTAS TROPICALES 500CC,POWERADE,0.5,L,True,2749,2749,24,24,"COTO CICSA, California Supermercados, Changoma...","Buenos Aires, CABA, Catamarca, Chaco, Chubut, ...",1700.00,3800.0,2195.17,0.166
1,7790895640025,POWERADE MOUNT BLAST 500CC,POWERADE,0.5,L,True,2723,2723,23,24,"COTO CICSA, California Supermercados, Changoma...","Buenos Aires, CABA, Catamarca, Chaco, Chubut, ...",225.08,3800.0,2197.88,0.167
2,7790580716707,SALADIX GALLETITA JAMON 100 GR,SALADIX,0.1,KG,True,2302,2302,23,24,"COTO CICSA, California Supermercados, Changoma...","Buenos Aires, CABA, Catamarca, Chaco, Chubut, ...",1339.00,2950.0,1778.91,0.164
3,7790064000261,ESTRELLA CLASICO 75 GR ALGODON,ESTRELLA,0.075,KG,True,2696,2696,22,24,"COTO CICSA, Changomas, Cooperativa Obrera Limi...","Buenos Aires, CABA, Catamarca, Chaco, Chubut, ...",947.99,1650.0,1388.11,0.060
4,7790895640476,AQ PERA 1500CC,AQUARIUS,1.5,L,True,2695,2695,22,24,"Axion Energy, COTO CICSA, Changomas, Cooperati...","Buenos Aires, CABA, Catamarca, Chaco, Chubut, ...",2345.00,5200.0,3089.69,0.168
5,7790895640018,POWERADE MANZANA 500 CC,POWERADE,0.5,L,True,2634,2634,22,24,"COTO CICSA, California Supermercados, Changoma...","Buenos Aires, CABA, Catamarca, Chaco, Chubut, ...",225.08,3400.0,2120.53,0.102
6,7798062548716,LEVITE MANZ 500CC,LEVITE,0.5,L,True,2560,2560,22,24,"Axion Energy, COTO CICSA, Changomas, Cooperati...","Buenos Aires, CABA, Catamarca, Chaco, Chubut, ...",1150.00,3400.0,2113.17,0.143
7,7790072002080,CELUSAL EST FIN 500G,CELUSAL,0.5,KG,True,2479,2479,22,24,"COTO CICSA, California Supermercados, Changoma...","Buenos Aires, CABA, Catamarca, Chaco, Chubut, ...",1183.99,2149.0,1804.58,0.058
8,7790064001909,ESTRELLA DISCOS DESM,ESTRELLA,80,EA,True,2255,2255,22,24,"COTO CICSA, Changomas, Cooperativa Obrera Limi...","Buenos Aires, CABA, Catamarca, Chaco, Chubut, ...",2090.00,3950.0,3266.49,0.072
9,7790580697303,SALADIX GALLETITA PIZZA 100 GR,SALADIX,0.1,KG,True,1857,1857,22,24,"COTO CICSA, California Supermercados, Changoma...","Buenos Aires, CABA, Catamarca, Chaco, Chubut, ...",1399.00,2950.0,1793.31,0.179


In [6]:
# ============================================================
# CELDA 6 — Cobertura por provincia y cadena + matriz
# ============================================================
cobertura_prov = pd.DataFrame([
    {'provincia': p,
     'n_productos_unicos': len(d['productos']),
     'n_sucursales': len(d['sucursales']),
     'n_cadenas': len(d['cadenas']),
     'n_registros': d['n_reg']}
    for p, d in cob_prov.items()
]).sort_values('n_productos_unicos', ascending=False).reset_index(drop=True)

cobertura_cadena = pd.DataFrame([
    {'cadena': c,
     'n_productos_unicos': len(d['productos']),
     'n_sucursales': len(d['sucursales']),
     'n_provincias': len(d['provincias']),
     'n_registros': d['n_reg']}
    for c, d in cob_cad.items()
]).sort_values('n_productos_unicos', ascending=False).reset_index(drop=True)

matriz_df = pd.DataFrame([
    {'cadena': k[0], 'provincia': k[1], 'n_registros': v}
    for k, v in matriz_cad_prov.items()
])
matriz = matriz_df.pivot_table(index='cadena', columns='provincia',
                                values='n_registros', fill_value=0)

print("Cobertura provincia:")
print(cobertura_prov)
print("\nCobertura cadena:")
print(cobertura_cadena)

Cobertura provincia:
              provincia  n_productos_unicos  n_sucursales  n_cadenas  \
0          Buenos Aires               75056          1025         21   
1                  CABA               62876          1166         16   
2               Córdoba               62382           183         16   
3              Santa Fe               60574            67         13   
4            Entre Ríos               57384            97         12   
5               Mendoza               56141            80         12   
6               Neuquén               54216            61          9   
7            Corrientes               49921            25          8   
8                Chubut               49815            49         10   
9             Río Negro               46508            69          7   
10              Tucumán               46389            28          8   
11                Salta               46276            52          9   
12                Chaco               45723

In [7]:
# ============================================================
# CELDA 7 — Exportar a Excel y descargar
# ============================================================
with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
    resumen.to_excel(writer, sheet_name='productos_unicos', index=False)
    cobertura_prov.to_excel(writer, sheet_name='cobertura_provincia', index=False)
    cobertura_cadena.to_excel(writer, sheet_name='cobertura_cadena', index=False)
    matriz.to_excel(writer, sheet_name='matriz_cadena_x_provincia')

    meta = pd.DataFrame({
        'campo': ['fecha_relevamiento','zips_procesados','filas_productos_procesadas',
                  'productos_unicos','cadenas_distintas','provincias_distintas'],
        'valor': [carpeta_dia, len(zips_cadenas), total_filas_productos,
                  len(resumen), len(cobertura_cadena), len(cobertura_prov)]
    })
    meta.to_excel(writer, sheet_name='metadata', index=False)

print(f"✅ Archivo generado: {OUTPUT_XLSX}")

from google.colab import files
files.download(OUTPUT_XLSX)

✅ Archivo generado: /content/productos_unicos_resumen.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# ============================================================
# CELDA 8 — Canasta valorada por provincia y ranking
# ============================================================
# Esta celda RECORRE OTRA VEZ los zips, pero filtrando solo los 30 EANs.
# Es liviano porque la canasta es chica. Acumula precios por (EAN, provincia).

from collections import defaultdict
import gc

# --- Definición de la canasta ---
# (EAN, nombre_corto, cantidad_mensual, categoria)
CANASTA = {
    '7790742363008': ('Leche entera 1L',         20, 'Lácteos'),
    '7791337007628': ('Yogur 190g',               8, 'Lácteos'),
    '7791337061361': ('Queso Casancrem 290g',     2, 'Lácteos'),
    '7793940052002': ('Manteca 100g',             2, 'Lácteos'),
    '7791337007253': ('Cindor 1L',                4, 'Lácteos'),
    '7790272001029': ('Aceite girasol 1,5L',      2, 'Almacén'),
    '7790070433114': ('Arroz 500g',               2, 'Almacén'),
    '7790070320285': ('Fideos 500g',              4, 'Almacén'),
    '7792180140708': ('Harina leudante 1kg',      2, 'Almacén'),
    '7792710000182': ('Yerba 500g',               2, 'Almacén'),
    '7790550000157': ('Café 250g',                1, 'Almacén'),
    '7790040143234': ('Chocolinas 250g',          4, 'Almacén'),
    '7790072002080': ('Sal fina 500g',            1, 'Almacén'),
    '7790895000232': ('Coca Cola lata',           8, 'Bebidas'),
    '7790895067570': ('Coca Sin Azúcar 2,25L',    4, 'Bebidas'),
    '7798062548716': ('Agua Levite 500ml',        8, 'Bebidas'),
    '7793147118860': ('Cerveza lata',             6, 'Bebidas'),
    '7798074864675': ('Vino Malbec 750ml',        2, 'Bebidas'),
    '7790132098459': ('Lavandina 1L',             2, 'Limpieza'),
    '7791290794054': ('Detergente 300ml',         2, 'Limpieza'),
    '7793253003500': ('Limpiador Poett 900ml',    2, 'Limpieza'),
    '7791293047447': ('Shampoo 400ml',            1, 'Higiene'),
    '7791293045948': ('Acondicionador 340ml',     1, 'Higiene'),
    '7791293051208': ('Jabón tocador 90g',        4, 'Higiene'),
    '7791293049557': ('Antitranspirante',         2, 'Higiene'),
    '7891024183083': ('Hilo dental',              1, 'Higiene'),
    '7790770601899': ('Toallas femeninas x16',    2, 'Higiene'),
    '7790250015840': ('Papel higiénico',          2, 'Higiene'),
    '7790580327415': ('Rocklets 40g',             2, 'Snacks'),
    '7790580716707': ('Saladix 100g',             2, 'Snacks'),
}

EANS_CANASTA = set(CANASTA.keys())
print(f"Canasta definida: {len(CANASTA)} productos")

# Acumulador: (ean, provincia) -> [suma_precios, n]
precios_ean_prov = defaultdict(lambda: [0.0, 0])

for i, zp in enumerate(zips_cadenas, 1):
    df_com = leer_csv_sepa(zp, 'comercio.csv', columnas=COLS_COM)
    df_suc = leer_csv_sepa(zp, 'sucursales.csv', columnas=COLS_SUC)
    df_pro = leer_csv_sepa(zp, 'productos.csv', columnas=COLS_PROD)
    if df_pro is None or df_suc is None or df_com is None:
        continue

    # Filtrar productos.csv solo a los EANs de la canasta
    df_pro = df_pro[df_pro['id_producto'].isin(EANS_CANASTA)]
    if len(df_pro) == 0:
        del df_com, df_suc, df_pro; gc.collect(); continue

    # Lookup sucursal -> provincia (solo provincia hace falta)
    suc_lookup = {}
    for r in df_suc.itertuples(index=False):
        suc_lookup[(r.id_comercio, r.id_bandera, r.id_sucursal)] = r.sucursales_provincia

    df_pro['precio'] = pd.to_numeric(df_pro['productos_precio_lista'], errors='coerce')

    for r in df_pro.itertuples(index=False):
        precio = r.precio
        if precio is None or pd.isna(precio) or precio <= 0:
            continue
        prov_iso = suc_lookup.get((r.id_comercio, r.id_bandera, r.id_sucursal))
        provincia = MAESTRO_PROV.get(prov_iso) if prov_iso else None
        if not provincia:
            continue
        key = (r.id_producto, provincia)
        precios_ean_prov[key][0] += precio
        precios_ean_prov[key][1] += 1

    del df_com, df_suc, df_pro
    gc.collect()

print(f"Combinaciones (EAN x provincia) con precios: {len(precios_ean_prov):,}")

Canasta definida: 30 productos
Combinaciones (EAN x provincia) con precios: 719


In [9]:
# ============================================================
# CELDA 9 — Construir tablas de precios y canasta valorada
# ============================================================
# Tabla larga: una fila por (EAN, provincia) con el precio promedio
filas = []
for (ean, prov), (suma, n) in precios_ean_prov.items():
    nombre, qty, cat = CANASTA[ean]
    precio_prom = suma / n
    filas.append({
        'id_producto': ean,
        'producto': nombre,
        'categoria': cat,
        'cantidad_mensual': qty,
        'provincia': prov,
        'precio_promedio': round(precio_prom, 2),
        'subtotal_canasta': round(precio_prom * qty, 2),
        'n_observaciones': n,
    })

precios_long = pd.DataFrame(filas)

# Pivot: una fila por producto, una columna por provincia (para vista rápida)
pivot_precios = precios_long.pivot_table(
    index=['id_producto','producto','categoria','cantidad_mensual'],
    columns='provincia',
    values='precio_promedio'
).round(2)

# Canasta total por provincia
canasta_provincia = (precios_long.groupby('provincia')
                     .agg(canasta_total=('subtotal_canasta','sum'),
                          productos_disponibles=('id_producto','nunique'),
                          n_observaciones=('n_observaciones','sum'))
                     .reset_index())
canasta_provincia['productos_faltantes'] = len(CANASTA) - canasta_provincia['productos_disponibles']

# Estimación "completa" para provincias con faltantes (regla simple):
# Si falta algún producto, completamos con el precio promedio nacional de ese producto.
prom_nacional = precios_long.groupby('id_producto')['precio_promedio'].mean()
total_qty_x_prom_nacional = sum(prom_nacional[ean] * CANASTA[ean][1]
                                 for ean in CANASTA if ean in prom_nacional.index)

def canasta_imputada(prov):
    sub = precios_long[precios_long['provincia']==prov]
    eans_presentes = set(sub['id_producto'])
    total = sub['subtotal_canasta'].sum()
    for ean, (nombre, qty, cat) in CANASTA.items():
        if ean not in eans_presentes and ean in prom_nacional.index:
            total += prom_nacional[ean] * qty
    return round(total, 2)

canasta_provincia['canasta_completa_imputada'] = canasta_provincia['provincia'].apply(canasta_imputada)

# RANKING (canasta completa imputada, de menor a mayor)
ranking = canasta_provincia.sort_values('canasta_completa_imputada').reset_index(drop=True)
ranking.insert(0, 'ranking', ranking.index + 1)
ranking['vs_promedio_pais_%'] = (
    (ranking['canasta_completa_imputada'] / ranking['canasta_completa_imputada'].mean() - 1) * 100
).round(2)

print("=== RANKING DE PROVINCIAS POR PRECIO DE CANASTA ===")
print(ranking[['ranking','provincia','canasta_completa_imputada',
               'productos_disponibles','productos_faltantes',
               'vs_promedio_pais_%']].to_string(index=False))

=== RANKING DE PROVINCIAS POR PRECIO DE CANASTA ===
 ranking           provincia  canasta_completa_imputada  productos_disponibles  productos_faltantes  vs_promedio_pais_%
       1             Tucumán                  316276.28                     30                    0               -4.00
       2             Córdoba                  316414.27                     30                    0               -3.96
       3 Santiago del Estero                  319386.21                     30                    0               -3.05
       4               Chaco                  320329.59                     30                    0               -2.77
       5          Entre Ríos                  322488.16                     30                    0               -2.11
       6             Formosa                  323499.22                     30                    0               -1.80
       7           Catamarca                  324759.69                     30                    0         

In [10]:
# ============================================================
# CELDA 10 — Exportar canasta a Excel
# ============================================================
OUTPUT_CANASTA = "/content/canasta_por_provincia.xlsx"

with pd.ExcelWriter(OUTPUT_CANASTA, engine='openpyxl') as writer:
    ranking.to_excel(writer, sheet_name='ranking_provincias', index=False)
    canasta_provincia.to_excel(writer, sheet_name='canasta_total', index=False)
    pivot_precios.to_excel(writer, sheet_name='precios_por_provincia')
    precios_long.to_excel(writer, sheet_name='detalle_largo', index=False)

    # Hoja con la definición de la canasta
    df_canasta = pd.DataFrame([
        {'id_producto': ean, 'producto': v[0], 'cantidad_mensual': v[1], 'categoria': v[2]}
        for ean, v in CANASTA.items()
    ])
    df_canasta.to_excel(writer, sheet_name='definicion_canasta', index=False)

print(f"✅ Archivo generado: {OUTPUT_CANASTA}")

from google.colab import files
files.download(OUTPUT_CANASTA)

✅ Archivo generado: /content/canasta_por_provincia.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>